# Phase 06A.03 — Zero-shot F1/F3/F8 validation
Evaluates the pinned model under a common maximum visual-tile budget. This notebook does not select a winner.

In [1]:
import gc,json,os,sys,time
from pathlib import Path
PROJECT_ROOT=Path('/workspace/RoadBuddy'); SRC_DIR=PROJECT_ROOT/'src'
if str(SRC_DIR) not in sys.path: sys.path.insert(0,str(SRC_DIR))
os.chdir(PROJECT_ROOT)
from roadbuddy_common import *
from phase06a_common import *
seed_everything(SEED)

## Configuration

In [2]:
RUN_SCOPE = 'full'  # temporary execution copy: smoke | full
SMOKE_LIMIT=20; FRAME_COUNTS=[1,3,8]
VALIDATION_CSV=PROJECT_ROOT/'data/splits/phase01/validation.csv'; VALIDATION_IDS=PROJECT_ROOT/'data/splits/phase01/validation_sample_ids.json'
AUDIT_STATUS=PROJECT_ROOT/f'outputs/phase06a/data_audit/{RUN_SCOPE}/PHASE06A_00_STATUS.json'
OUTPUT_ROOT=PROJECT_ROOT/'outputs/phase06a/zero_shot'
protocol=Phase06AProtocol(run_scope=RUN_SCOPE); limit=SMOKE_LIMIT if RUN_SCOPE=='smoke' else None
assert VALIDATION_CSV.is_file() and VALIDATION_IDS.is_file() and AUDIT_STATUS.is_file()
audit=json.loads(AUDIT_STATUS.read_text(encoding='utf-8'))
if RUN_SCOPE=='full': assert audit['status']=='PASS' and audit['hard_gates_passed']

## Frozen membership and pinned model

In [3]:
val_df=pd.read_csv(VALIDATION_CSV); frozen_ids=json.loads(VALIDATION_IDS.read_text(encoding='utf-8'))
assert sorted(val_df.sample_id.astype(str))==sorted(map(str,frozen_ids))
if RUN_SCOPE=='full': assert len(val_df)==EXPECTED_VALIDATION_ROWS
model,tokenizer=load_model_and_tokenizer(training=False,attn_implementation=protocol.attention_implementation)
model.img_context_token_id=tokenizer.convert_tokens_to_ids(IMG_CONTEXT_TOKEN); assert model.template=='Hermes-2'

/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


FlashAttention2 is not installed.


## Run independently checkpointed frame arms

In [4]:
summaries=[]
for frame_count in FRAME_COUNTS:
    experiment_id=f'Z-F{frame_count}'; out_dir=OUTPUT_ROOT/experiment_id/RUN_SCOPE; out_dir.mkdir(parents=True,exist_ok=True)
    started=time.perf_counter()
    predictions,metrics=evaluate_fixed_budget_rows(model,tokenizer,val_df,frame_count=frame_count,total_tile_budget=protocol.total_tile_budget,limit=limit)
    expected_ids=val_df.head(limit).sample_id if limit else frozen_ids
    integrity=validate_prediction_artifact(predictions,expected_ids,run_scope=RUN_SCOPE)
    predictions.to_csv(out_dir/'predictions.csv',index=False)
    config={**protocol.to_dict(),'experiment_id':experiment_id,'frame_count':frame_count,'limit':limit,'model_id':MODEL_ID,'model_revision':MODEL_REVISION}
    save_json(out_dir/'config.json',config); save_json(out_dir/'metrics.json',metrics)
    runtime={'elapsed_seconds':time.perf_counter()-started,'peak_vram_gib':torch.cuda.max_memory_allocated()/1024**3}; save_json(out_dir/'runtime.json',runtime)
    write_run_manifest(out_dir/'run_manifest.json',config=config,artifacts=[out_dir/'predictions.csv',out_dir/'metrics.json',out_dir/'runtime.json'])
    summaries.append({'experiment_id':experiment_id,**metrics,'integrity':integrity,'runtime':runtime})
summary=pd.DataFrame([{k:v for k,v in row.items() if k not in {'per_class','confusion_matrix','integrity','runtime'}} for row in summaries])
summary.to_csv(OUTPUT_ROOT/f'zero_shot_summary_{RUN_SCOPE}.csv',index=False)
save_json(OUTPUT_ROOT/f'PHASE06A_03_STATUS_{RUN_SCOPE}.json',{'phase':'06A.03','status':'complete' if RUN_SCOPE=='full' else 'smoke_complete','experiments':[f'Z-F{x}' for x in FRAME_COUNTS],'rows_per_experiment':len(val_df) if limit is None else limit})

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


...

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


## Cleanup and interpretation constraint

In [5]:
del model; gc.collect(); torch.cuda.empty_cache(); display(summary)

,experiment_id,rows,accuracy,macro_f1,parse_rate
0,Z-F1,298,0.439597,0.394899,0.996644
1,Z-F3,298,0.483221,0.444526,1.000000
2,Z-F8,298,0.486577,0.448848,1.000000


Frame-arm differences remain validation evidence only. Winner selection and statistical correction occur exclusively in Phase 06A.07.